# GeoSR-4 — DINOv3 sat493m perceptual-backbone run (Colab GPU)

Standalone notebook -- only the DINOv3 sat493m run (D042), no DINOv2 section, so this can run in a fresh Colab session/account without repeating the DINOv2 run first.

`facebook/dinov3-vitl16-pretrain-sat493m` is ViT-L (300M params), satellite-pretrained (SAT-493M dataset) -- gated access, approved for this project already (D042). `--batch-size 4` is a conservative starting guess against T4's 15GB (this model is much bigger than DINOv2-small's 21M params) -- reduce further if it OOMs, see D024/D025 for the same kind of VGG-perceptual-loss OOM fix pattern.

Config otherwise matches D028/D029's VGG "quality run" protocol as closely as possible (30 epochs, lr 1e-4, lambda-perceptual 0.01, ICNR on) so eval numbers stay comparable to D028 (VGG) and the DINOv2 run already done separately.

**~9 min/epoch observed -- the full 30 epochs is ~4.5 hours.** A previous run on another account hit a Colab usage-limit disconnect at epoch 24 and the checkpoints were lost with the ephemeral VM (D042). This notebook now mounts Google Drive (section 2a) so checkpoints survive a disconnect, and `train_swinir.py` now supports `--resume-from` (section 4b) so a disconnect costs you the remaining epochs, not all of them.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
`transformers` is needed to load the DINOv3 backbone.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision transformers

## 2a. Mount Google Drive (so checkpoints survive a disconnect)
Run this once. It'll ask for Drive permission -- approve it. Checkpoints go to
`/content/drive/MyDrive/geosr4_checkpoints/swinir_dino3_sat_quality/` instead of the ephemeral Colab VM disk,
so if this session gets disconnected (usage limit, timeout, etc.) the checkpoints already saved are still
there when you start a new session -- pass the last one to `--resume-from` (section 4b) instead of restarting
from epoch 0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/geosr4_checkpoints/swinir_dino3_sat_quality"
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints will be saved to", CKPT_DIR)

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Hugging Face login (required -- gated model)
This opens an interactive prompt -- paste your HF **token** there (never in a code cell/chat). Must be logged in as the account that was granted access to `facebook/dinov3-vitl16-pretrain-sat493m` -- if you have multiple HF accounts, generate the token from **that specific account's** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (Read scope is enough), rather than using the device-code login flow (which can silently log you into the wrong account if your browser has another HF session active).

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## 4. Train: DINOv3 sat493m perceptual backbone
Checkpoints go to `CKPT_DIR` (on Drive, from section 2a) instead of local Colab disk -- if this session
disconnects partway through, the checkpoints already written are safe. If you're continuing after a disconnect
rather than starting fresh, skip this cell and go to **4b** instead.

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir {CKPT_DIR} \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint {CKPT_DIR}/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 4b. Resuming after a disconnect (skip this if section 4 just ran fine)
Only run this if a previous attempt got disconnected partway through -- it picks up from the last checkpoint
saved on Drive instead of restarting at epoch 0. Note: only the model weights resume, not the optimizer's
internal momentum state (a documented simplification, see `--resume-from`'s help text in `train_swinir.py`) --
fine for finishing an interrupted run, not meant as a precision-training feature.

In [ ]:
import glob, re

ckpts = glob.glob(f"{CKPT_DIR}/swinir_epoch*.pt")
last_ckpt = max(ckpts, key=lambda p: int(re.search(r"epoch(\d+)", p).group(1)))
print("resuming from", last_ckpt)

!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 4 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --perceptual-backbone dino \
  --dino-model-id facebook/dinov3-vitl16-pretrain-sat493m \
  --amp \
  --checkpoint-dir {CKPT_DIR} \
  --resume-from {last_ckpt} \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint {CKPT_DIR}/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Download the checkpoint

In [ ]:
from google.colab import files
files.download(f"{CKPT_DIR}/swinir_epoch29.pt")